In [9]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("../data/obesity_challenge_2.h5ad")

adata_genes = adata.obs["gene"].astype(str)

adata_set = set(adata_genes)

print(f"# perturbations in adata: {len(adata_set)}")
print(list(adata_set)[:10])


# 讀取 txt (每行一個 perturbation)
with open("../data/predict_perturbations_2.txt") as f:
    predict_genes = [line.strip() for line in f if line.strip()]

predict_set = set(predict_genes)

all_perts = adata_set.union(predict_set)

print("# total perturbations:", len(all_perts))

# perturbations in adata: 237
['TCF7L2', 'SF3B1+SF3B1+ZBED3', 'CEBPD+ZBED3', 'CEBPB+CEBPB+TCF7L2', 'FOXO1+STAT5A', 'KIF11+KLF15+NR3C1+NR3C1', 'KIF11+TCF7L2', 'CEBPD+KLF15', 'KLF15+TCF7L2+ZBED3', 'CEBPA+NC+ZBED3']
# total perturbations: 299


# GO embeddings
use PerturbNet

In [10]:
import numpy as np
import scipy.sparse as sp

# load files
G_sparse = sp.load_npz("../data/GO/sparse_gene_anno_matrix.npz")

genes = np.load("../data/GO/gene.npy", allow_pickle=True)

go_terms = np.load("../data/GO/anno.npy", allow_pickle=True)

gene_to_idx = {g:i for i,g in enumerate(genes)}
print(G_sparse.shape)

(18832, 15988)


In [13]:
from perturbnet.genotypevae.genotypeVAE import *

device = 'mps' if torch.cuda.is_available() else 'cpu'
path_genovae_model = "../data/GO/model_params.pt"
model_genovae = GenotypeVAE().to(device)
model_genovae.load_state_dict(torch.load(path_genovae_model, map_location = device))
model_genovae.eval()

GenotypeVAE(
  (linear_1): Linear(in_features=15988, out_features=512, bias=True)
  (linear_2): Linear(in_features=512, out_features=256, bias=True)
  (linear_3_mu): Linear(in_features=256, out_features=10, bias=True)
  (linear_3_std): Linear(in_features=256, out_features=10, bias=True)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear_4): Linear(in_features=10, out_features=256, bias=True)
  (linear_5): Linear(in_features=256, out_features=512, bias=True)
  (linear_6): Linear(in_features=512, out_features=15988, bias=True)
  (bn4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (softmax): Softmax(dim=None)
  (leaky): LeakyReLU(negative_slope=0.2)
  (sigmoid): Sigmoid()
  (dropout1): Dropout(p=0.2, inplace=F

In [20]:
import numpy as np
import torch

def build_perturb_embedding_dict(
    perturbations,
    G_sparse,
    gene_to_idx,
    model_genovae,
    device="cpu"
):

    perturb_to_embedding = {}

    for p in sorted(perturbations):

        genes = p.split("+")

        go_vecs = []

        for g in genes:

            if g == "NC":

                go_vecs.append(
                    np.zeros(G_sparse.shape[1])
                )

            elif g == "PPARG2":

                go_vecs.append(
                    G_sparse[
                        gene_to_idx["PPARG"]
                    ].toarray().flatten()
                )

            elif g in gene_to_idx:

                go_vecs.append(
                    G_sparse[
                        gene_to_idx[g]
                    ].toarray().flatten()
                )

            else:

                print("missing:", g)

                go_vecs.append(
                    np.zeros(G_sparse.shape[1])
                )

        # union GO
        go_union = np.clip(
            np.sum(go_vecs, axis=0),
            0,
            1
        )

        # encode
        x = torch.tensor(
            go_union,
            dtype=torch.float32
        ).unsqueeze(0).to(device)

        with torch.no_grad():

            mu, logvar = model_genovae.encode(x)

        perturb_to_embedding[p] = (
            mu.squeeze(0)
            .cpu()
            .numpy()
        )

    return perturb_to_embedding

In [21]:
all_perts = adata_set.union(predict_set)

perturb_to_embedding = build_perturb_embedding_dict(
    all_perts,
    G_sparse,
    gene_to_idx,
    model_genovae,
    device
)

In [22]:
perturb_to_embedding

{'CEBPA+CEBPA': array([-0.0283025 ,  0.00278124,  0.19738948, -0.01252955,  0.05589449,
        -0.05670704,  0.29455206, -0.13584971, -0.11629143,  0.1660095 ],
       dtype=float32),
 'CEBPA+CEBPA+CEBPB': array([-0.15122515,  0.01420031,  0.25806898, -0.02588699,  0.06289224,
        -0.2725547 ,  0.27127305, -0.00725004, -0.03099433,  0.3412358 ],
       dtype=float32),
 'CEBPA+CEBPA+NR3C1': array([-0.07719333, -0.00123557,  0.19157001, -0.08634515, -0.00621319,
        -0.01176301,  0.5543519 , -0.17625898, -0.17395827,  0.0593964 ],
       dtype=float32),
 'CEBPA+CEBPA+POLR2D': array([-0.07194245, -0.11195773,  0.13124004,  0.14695193, -0.30340028,
        -0.2300086 ,  0.53685737,  0.04719713, -0.06787631,  0.08072067],
       dtype=float32),
 'CEBPA+CEBPA+STAT5B': array([-0.23985578, -0.11453021,  0.15719247,  0.05306036,  0.01409796,
        -0.31312487,  0.45243692, -0.16873783, -0.16259357,  0.12666672],
       dtype=float32),
 'CEBPA+CEBPB': array([-0.15122515,  0.01420031, 

In [23]:
import pickle

with open(
    "../resources/perturbation_embeddings_genotypeVAE.pickle",
    "wb"
) as fp:

    pickle.dump(
        perturb_to_embedding,
        fp
    )

# Summmary GPT embeddings